# 04 – LSTM Model

This notebook trains a multi-layer LSTM network on the scaled, feature-engineered
dataset and evaluates it on the test set.

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from src.utils.data_loader import load_processed_data
from src.utils.metrics import calculate_metrics
from src.data.preprocessor import DataPreprocessor
from src.models.lstm_model import LSTMModel
from src.visualization.plotter import Plotter
from src.config import RESULTS_DIR, LSTM_SEQUENCE_LENGTH

plotter = Plotter()

In [2]:
# ── Load data & scale ───────────────────────────────────────────────────────
train = load_processed_data('train')
val   = load_processed_data('val')
test  = load_processed_data('test')

pre = DataPreprocessor()
t_sc, v_sc, te_sc = pre.scale_features(train, val, test)

target_col = 'Close'
target_idx = list(train.columns).index(target_col)

print(f'Target column index: {target_idx}')
print(f'Scaled shapes – train: {t_sc.shape}, val: {v_sc.shape}, test: {te_sc.shape}')

Target column index: 1
Scaled shapes – train: (1006, 32), val: (252, 32), test: (249, 32)


In [3]:
# ── Train LSTM ──────────────────────────────────────────────────────────────
lstm = LSTMModel(
    sequence_length=LSTM_SEQUENCE_LENGTH,
    lstm_units=[64, 32],
    dropout=0.2,
    epochs=50,
    batch_size=32,
)

lstm.fit(
    train_data=t_sc.values,
    val_data=v_sc.values,
    target_idx=target_idx,
)

print('LSTM training complete.')

LSTM training complete.


In [4]:
# ── Training curves ─────────────────────────────────────────────────────────
plotter.plot_loss_history(lstm.history, filename='lstm_training_loss.png')
print('Training loss plot saved.')

Training loss plot saved.


In [5]:
# ── Predict on test set ─────────────────────────────────────────────────────
# Combine val + test for context window
combined_scaled = np.vstack([v_sc.values, te_sc.values])
preds_scaled_raw = lstm.predict(combined_scaled, target_idx=target_idx)

# The LSTM needs SEQ_LEN rows of context before predicting the test period.
# The first prediction aligns with index SEQ_LEN in combined_scaled.
# We want predictions for test rows:
n_val  = len(v_sc)
n_test = len(te_sc)
# predictions start at combined position SEQ_LEN; test starts at n_val.
# index in preds_scaled_raw for first test day: n_val - SEQ_LEN (if n_val >= SEQ_LEN)
start = max(0, n_val - LSTM_SEQUENCE_LENGTH)
lstm_preds_scaled = preds_scaled_raw[start: start + n_test]

# Inverse-transform: reconstruct full scaled array, replace target, inverse-transform
te_inv = te_sc.copy()
te_inv.iloc[:len(lstm_preds_scaled), target_idx] = lstm_preds_scaled
te_orig = pre.inverse_scale(te_inv)
lstm_preds = te_orig.iloc[:len(lstm_preds_scaled), target_idx].values

y_test = test[target_col].values[:len(lstm_preds)]

lstm_metrics = calculate_metrics(y_test, lstm_preds)
print('LSTM metrics:', lstm_metrics)

LSTM metrics: {'mse': 6548250.482820664, 'rmse': 2558.9549591230916, 'mae': 2272.272313151961, 'mape': 9.686196854073852, 'directional_accuracy': 0.5282258064516129}


In [6]:
# ── Plot predictions ─────────────────────────────────────────────────────────
plotter.plot_predictions_comparison(
    y_test,
    {'LSTM': lstm_preds},
    dates=test.index[:len(lstm_preds)],
    title='LSTM Predictions vs Actual (Test Set)',
    filename='lstm_predictions.png'
)
print('Predictions plot saved.')

Predictions plot saved.


In [7]:
# ── Save model & predictions ─────────────────────────────────────────────────
lstm.save("lstm_model.keras")

lstm_df = pd.DataFrame(
    {'actual': y_test, 'lstm': lstm_preds},
    index=test.index[:len(lstm_preds)]
)
lstm_df.to_csv(RESULTS_DIR / 'lstm_predictions.csv')

import json
with open(RESULTS_DIR / 'lstm_metrics.json', 'w') as f:
    json.dump(lstm_metrics, f, indent=2)

print('Model and predictions saved.')

Model and predictions saved.


## Summary

LSTM model results are saved to `reports/results/lstm_metrics.json`.

Continue to **05_ensemble.ipynb** to combine all model predictions.